# Bloco 02 — Transformações Essenciais

## Aula 08 — PySpark com Northwind

Neste bloco você vai dominar as operações de transformação de colunas, filtros e manipulação de dados em PySpark.

**O que vamos aprender:**
1. Operações básicas: `select`, `filter`, `withColumn`, `withColumnRenamed`, `drop`, `distinct`
2. Funções essenciais: `col()`, `lit()`, `when().otherwise()`, `cast()`
3. Operações com strings: `upper()`, `lower()`, `trim()`, `concat()`, `split()`, `regexp_replace()`
4. Operações com datas: `year()`, `month()`, `datediff()`, `months_between()`
5. Tratamento de nulos: `isNull()`, `coalesce()`, `fillna()`, `dropna()`

**Pré-requisito:** Você já completou o Bloco 01 e sabe usar `spark.table()`, `printSchema()`, `display()`, etc.

---

## Setup

In [0]:
# pyspark.sql.functions — modulo principal de funcoes do PySpark.
# Contem funcoes para manipulacao de colunas, agregacoes, strings, datas, etc.
# Importamos como "F" por convencao para evitar conflito com built-ins do Python
# (ex: F.col(), F.sum(), F.max() em vez de col(), sum(), max()).
from pyspark.sql import functions as F

# pyspark.sql.types — modulo de tipos de dados do PySpark.
# Contem StructType, StructField, StringType, IntegerType, FloatType, etc.
# Aqui importamos tudo (*) porque vamos usar varios tipos ao longo do notebook.
from pyspark.sql.types import *

CATALOG = "northwind"
SCHEMA = "bronze"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# Carregar tabelas
orders = spark.table("orders")
order_details = spark.table("order_details")
customers = spark.table("customers")
products = spark.table("products")
employees = spark.table("employees")

---

## 1. Operações Básicas

Estas são as operações fundamentais que você vai usar em praticamente todo código PySpark.
Você já sabe fazer isso em SQL — agora vai aprender a sintaxe equivalente no DataFrame API.

### 1.1 `select()` — Selecionar colunas

Duas formas de usar:
- Com **strings**: `df.select("col1", "col2")`
- Com **col()**: `df.select(F.col("col1"), F.col("col2"))`

A forma com `col()` permite encadear operações (alias, cast, etc).

In [0]:
# select com strings
products.select("product_name", "unit_price").show(5)

In [0]:
# select com col() — permite encadear .alias(), .cast(), etc.
products.select(
    F.col("product_name"),
    F.col("unit_price").alias("preco_unitario")
).show(5)

### 1.2 `filter()` / `where()` — Filtrar linhas

`filter()` e `where()` são **idênticos** — use o que preferir.

Equivalente SQL: `WHERE`

In [0]:
# Filtrar pedidos de 1997
orders_1997 = orders.filter(F.year(F.col("order_date")) == 1997)

print(f"Total de pedidos em 1997: {orders_1997.count()}")
orders_1997.select("order_id", "customer_id", "order_date").show(5)

In [0]:
# Mesmo resultado usando where() e condições combinadas
# Filtrar pedidos de 1997 com frete > 50
orders.where(
    (F.year(F.col("order_date")) == 1997) & (F.col("freight") > 50)
).select("order_id", "order_date", "freight").show(5)

### 1.3 `withColumn()` — Criar ou modificar colunas

Cria uma **nova coluna** ou **substitui** uma existente.

**Pegadinha:** `withColumn()` retorna um **novo** DataFrame. O original não é modificado (imutabilidade).

In [0]:
# Criar coluna receita em order_details
order_details_receita = order_details.withColumn(
    "receita",
    F.col("unit_price") * F.col("quantity") * (F.lit(1.0) - F.col("discount"))
)

order_details_receita.select(
    "order_id", "product_id", "unit_price", "quantity", "discount", "receita"
).show(5)

### 1.4 `withColumnRenamed()` — Renomear colunas

Equivalente SQL: `SELECT col AS novo_nome`

In [0]:
# Renomear colunas para português
products_renamed = (
    products
    .withColumnRenamed("product_name", "nome_produto")
    .withColumnRenamed("unit_price", "preco_unitario")
    .withColumnRenamed("units_in_stock", "estoque")
)

products_renamed.select("nome_produto", "preco_unitario", "estoque").show(5)

### 1.5 `drop()` — Remover colunas

Remove uma ou mais colunas do DataFrame.

In [0]:
# Remover coluna fax de customers (coluna quase toda nula)
customers_clean = customers.drop("fax")

print("Colunas originais:", customers.columns)
print("Colunas sem fax:  ", customers_clean.columns)

### 1.6 `distinct()` / `dropDuplicates()` — Valores únicos

- `distinct()`: retorna linhas únicas considerando **todas** as colunas
- `dropDuplicates([cols])`: permite escolher quais colunas usar na deduplicacao

In [0]:
# Países distintos dos clientes
paises = customers.select("country").distinct().orderBy("country")

print(f"Total de países: {paises.count()}")
paises.show(10)

In [0]:
# dropDuplicates — manter primeiro registro por país
customers.dropDuplicates(["country"]).select(
    "customer_id", "company_name", "country"
).orderBy("country").show(5)


---

## 2. Funções Essenciais

As funções do módulo `pyspark.sql.functions` (importado como `F`) são o coração do PySpark.

### 2.1 `col()` e `lit()`

- `F.col("nome")` — referencia uma **coluna** do DataFrame
- `F.lit(valor)` — cria uma coluna com um **valor literal** (constante)

`lit()` é essencial quando você precisa misturar colunas com constantes em expressões.

In [0]:
# col() referencia uma coluna, lit() cria um valor constante
products.select(
    F.col("product_name"),
    F.col("unit_price"),
    F.lit("BRL").alias("moeda"),
    (F.col("unit_price") * F.lit(5.5)).alias("preco_brl")
).show(5)

### 2.2 `when().otherwise()` — Lógica condicional

Equivalente SQL: `CASE WHEN ... THEN ... ELSE ... END`

Permite criar colunas com valores condicionais.

In [0]:
# Classificar produtos como Caro, Moderado ou Barato
products_classificados = products.withColumn(
    "faixa_preco",
    F.when(F.col("unit_price") >= 50, "Caro")
     .when(F.col("unit_price") >= 20, "Moderado")
     .otherwise("Barato")
)

products_classificados.select(
    "product_name", "unit_price", "faixa_preco"
).orderBy(F.col("unit_price").desc()).show(10)

In [0]:
# Classificar produtos como ativo/descontinuado
products.withColumn(
    "status",
    F.when(F.col("discontinued") == 1, "Descontinuado")
     .otherwise("Ativo")
).groupBy("status").count().show()

### 2.3 `cast()` — Conversão de tipos

Equivalente SQL: `CAST(col AS tipo)`

Duas formas de usar:
- `F.col("coluna").cast("integer")`
- `F.col("coluna").cast(IntegerType())`

In [0]:
# cast() — converter tipos
order_details.select(
    "order_id",
    F.col("unit_price").cast("decimal(10,2)").alias("preco_decimal"),
    F.col("quantity").cast("integer").alias("qtd_int"),
    F.col("discount").cast("decimal(4,2)").alias("desconto_decimal")
).show(5)

# Verificar tipos
print("Tipo original de unit_price:", order_details.schema["unit_price"].dataType)
print("Tipo original de quantity:  ", order_details.schema["quantity"].dataType)

### 2.4 Fórmula Central: Receita

A fórmula de receita do Northwind é o cálculo mais importante do dataset:

```
receita = unit_price * quantity * (1.0 - discount)
```

Vamos calcular usando `col()` e `lit()` juntos.

In [0]:
# Formula central de receita do Northwind
order_details_receita = order_details.withColumn(
    "receita",
    F.col("unit_price") * F.col("quantity") * (F.lit(1.0) - F.col("discount"))
)

# Exibir com receita formatada
order_details_receita.select(
    "order_id",
    "product_id",
    "unit_price",
    "quantity",
    "discount",
    F.round("receita", 2).alias("receita")
).show(10)

# Receita total do Northwind
total = order_details_receita.agg(F.round(F.sum("receita"), 2).alias("receita_total"))
total.show()

---

## 3. Operações com Strings

Funções para manipular colunas de texto. Muito úteis para padronização e limpeza de dados.

### 3.1 `upper()`, `lower()`, `trim()` — Padronização

Essenciais para padronizar nomes de países, cidades, etc.

In [0]:
# Padronizar nomes de países para maiúsculo
customers_padronizado = customers.withColumn(
    "country_upper", F.upper(F.col("country"))
).withColumn(
    "city_lower", F.lower(F.col("city"))
).withColumn(
    "company_trimmed", F.trim(F.col("company_name"))
)

customers_padronizado.select(
    "country", "country_upper", "city", "city_lower"
).distinct().orderBy("country").show(10)

### 3.2 `concat()` — Concatenar strings

Use `concat()` ou `concat_ws()` (com separador).

In [0]:
# Criar nome completo dos funcionários
employees_nome_completo = employees.withColumn(
    "full_name",
    F.concat_ws(" ", F.col("first_name"), F.col("last_name"))
)

employees_nome_completo.select(
    "employee_id", "first_name", "last_name", "full_name", "title"
).show(truncate=False)

In [0]:
# concat() sem separador — criar código de cliente
customers.select(
    "customer_id",
    F.concat(
        F.upper(F.col("country")),
        F.lit("-"),
        F.col("customer_id")
    ).alias("customer_code")
).show(5, truncate=False)

### 3.3 `split()` e `substring()` — Extrair partes de strings

In [0]:
# split() — quebrar endereço por vírgula
customers.select(
    "customer_id",
    "address",
    F.split(F.col("address"), " ").getItem(0).alias("primeiro_token")
).show(5, truncate=False)

In [0]:
# substring() — extrair os 3 primeiros caracteres do customer_id
customers.select(
    "customer_id",
    F.substring(F.col("customer_id"), 1, 3).alias("prefixo")
).show(5)

### 3.4 `regexp_replace()` — Limpeza com regex

In [0]:
# Remover caracteres especiais do telefone (parênteses, hifens, espaços)
customers.select(
    "customer_id",
    "phone",
    F.regexp_replace(F.col("phone"), r"[\(\)\-\s\.]", "").alias("phone_clean")
).show(10, truncate=False)

### 3.5 `explode()` — Transformar arrays em linhas

`explode()` pega uma coluna do tipo **array** e cria **uma linha para cada elemento**. E o inverso de `collect_list()`.

In [0]:
# Criar array a partir de split — ex: quebrar address por espaço
customers_com_array = customers.select(
    "customer_id",
    "address",
    F.split(F.col("address"), " ").alias("palavras_endereco")
)

customers_com_array.show(3, truncate=False)

# explode() — cada palavra vira uma linha separada
customers_exploded = customers_com_array.select(
    "customer_id",
    F.explode(F.col("palavras_endereco")).alias("palavra")
)

customers_exploded.show(10, truncate=False)
print(f"Antes do explode: {customers_com_array.count()} linhas")
print(f"Depois do explode: {customers_exploded.count()} linhas")

In [0]:
# Caso prático: collect_list (inverso do explode)
# Agrupar todos os produtos comprados por cada pedido em uma lista
order_products = (
    order_details
    .join(products, "product_id")
    .groupBy("order_id")
    .agg(F.collect_list("product_name").alias("produtos"))
)

order_products.show(5, truncate=False)

---

## 4. Operações com Datas

O Northwind tem datas ricas para explorar: `order_date`, `shipped_date`, `required_date`, `hire_date`, `birth_date`.

### 4.1 `year()`, `month()`, `dayofweek()` — Extrair componentes

In [0]:
# Extrair ano, mês e dia da semana de order_date
orders_datas = orders.select(
    "order_id",
    "order_date",
    F.year(F.col("order_date")).alias("ano"),
    F.month(F.col("order_date")).alias("mes"),
    F.dayofweek(F.col("order_date")).alias("dia_semana"),
    F.dayofmonth(F.col("order_date")).alias("dia_mes")
)

orders_datas.show(10)

In [0]:
# Quantos pedidos por dia da semana?
# dayofweek: 1=Domingo, 2=Segunda, ..., 7=Sábado
orders_datas.groupBy("dia_semana").count().orderBy("dia_semana").show()

### 4.2 `to_date()` e `date_format()` — Converter e formatar datas

In [0]:
# date_format() — formatar datas como string
orders.select(
    "order_id",
    "order_date",
    F.date_format(F.col("order_date"), "dd/MM/yyyy").alias("data_br"),
    F.date_format(F.col("order_date"), "yyyy-MM").alias("ano_mes"),
    F.date_format(F.col("order_date"), "EEEE").alias("dia_semana_nome")
).show(5, truncate=False)

### 4.3 `datediff()` — Diferença entre datas

Fundamental para calcular **dias para entrega** no Northwind:
- `datediff(shipped_date, order_date)` = dias entre pedido e entrega

In [0]:
# Calcular dias para entrega
orders_entrega = orders.withColumn(
    "dias_para_entrega",
    F.datediff(F.col("shipped_date"), F.col("order_date"))
)

orders_entrega.select(
    "order_id", "order_date", "shipped_date", "dias_para_entrega"
).show(10)

# Estatísticas de tempo de entrega
orders_entrega.select(
    F.round(F.avg("dias_para_entrega"), 1).alias("media_dias"),
    F.min("dias_para_entrega").alias("min_dias"),
    F.max("dias_para_entrega").alias("max_dias")
).show()

### 4.4 `months_between()` — Meses entre datas

In [0]:
# Tempo de empresa dos funcionários (em meses desde a contratação)
employees.select(
    "employee_id",
    F.concat_ws(" ", F.col("first_name"), F.col("last_name")).alias("nome"),
    "hire_date",
    F.round(
        F.months_between(F.current_date(), F.col("hire_date")), 0
    ).cast("integer").alias("meses_empresa"),
    F.round(
        F.months_between(F.current_date(), F.col("hire_date")) / 12, 1
    ).alias("anos_empresa")
).orderBy("hire_date").show(truncate=False)

### 4.5 `current_date()` — Data atual

In [0]:
# Quantos dias desde o último pedido no Northwind?
orders.select(
    F.max("order_date").alias("ultimo_pedido"),
    F.current_date().alias("hoje"),
    F.datediff(
        F.current_date(), F.max("order_date")
    ).alias("dias_desde_ultimo_pedido")
).show()

---

## 5. Tratamento de Nulos

Dados reais sempre têm nulos. O Northwind tem vários:
- `shipped_date`: NULL quando o pedido ainda não foi enviado
- `region`: muitos clientes sem região preenchida
- `fax`: quase todos nulos
- `reports_to`: gerente geral não tem reports_to

### 5.1 `isNull()` / `isNotNull()` — Identificar nulos

In [0]:
# Pedidos que ainda não foram enviados (shipped_date é NULL)
pedidos_pendentes = orders.filter(F.col("shipped_date").isNull())

print(f"Pedidos pendentes (sem shipped_date): {pedidos_pendentes.count()}")
pedidos_pendentes.select(
    "order_id", "customer_id", "order_date", "shipped_date"
).show()

In [0]:
# Pedidos que já foram enviados
pedidos_enviados = orders.filter(F.col("shipped_date").isNotNull())

print(f"Pedidos enviados: {pedidos_enviados.count()}")
print(f"Pedidos totais:   {orders.count()}")

In [0]:
# Contar nulos em cada coluna de customers
customers.select(
    [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in customers.columns]
).show(truncate=False)

### 5.2 `coalesce()` — Valor padrão para nulos

`coalesce()` retorna o **primeiro valor não-nulo** de uma lista de colunas.

Equivalente SQL: `COALESCE(col1, col2, 'default')`

In [0]:
# Tratar region nula com coalesce — substituir NULL por "N/A"
customers_region = customers.withColumn(
    "region_tratada",
    F.coalesce(F.col("region"), F.lit("N/A"))
)

# Mostrar antes e depois
customers_region.select(
    "customer_id", "company_name", "country", "region", "region_tratada"
).show(15, truncate=False)

### 5.3 `fillna()` / `na.fill()` — Preencher nulos em massa

Preenche nulos em várias colunas de uma vez.

In [0]:
# Preencher nulos de region e fax de uma vez
customers_filled = customers.fillna({
    "region": "N/A",
    "fax": "Sem fax"
})

# Equivalente: customers.na.fill({"region": "N/A", "fax": "Sem fax"})

customers_filled.select(
    "customer_id", "region", "fax"
).show(10, truncate=False)

### 5.4 `dropna()` / `na.drop()` — Remover linhas com nulos

In [0]:
# dropna() — remover linhas que tenham qualquer nulo
print(f"Customers total:           {customers.count()}")
print(f"Customers sem nenhum nulo: {customers.dropna().count()}")

# Remover apenas linhas com nulo em colunas específicas
print(f"Customers com region preenchida: {customers.dropna(subset=['region']).count()}")

# Usar thresh — manter linhas com pelo menos N valores não-nulos
# Exemplo: manter se pelo menos 9 das 11 colunas estiverem preenchidas
print(f"Customers com >= 9 colunas preenchidas: {customers.dropna(thresh=9).count()}")

---

## Exercícios

Agora é sua vez! Aplique o que aprendeu neste bloco.

### Exercício 1 — Receita total e filtro

Usando `order_details`:
1. Crie a coluna `receita_total` com a fórmula: `unit_price * quantity * (1.0 - discount)`
2. Filtre apenas as linhas com `receita_total > 1000`
3. Ordene por `receita_total` descendente
4. Mostre as 10 primeiras linhas

In [0]:
# Exercicio 1 — Solução
exercicio_1 = (
    order_details
    .withColumn(
        "receita_total",
        F.col("unit_price") * F.col("quantity") * (F.lit(1.0) - F.col("discount"))
    )
    .filter(F.col("receita_total") > 1000)
    .orderBy(F.col("receita_total").desc())
)

print(f"Itens com receita > 1000: {exercicio_1.count()}")
exercicio_1.select(
    "order_id", "product_id", "unit_price", "quantity", "discount",
    F.round("receita_total", 2).alias("receita_total")
).show(10)

### Exercício 2 — Colunas de data e tempo de entrega

Usando `orders`:
1. Crie a coluna `ano` extraindo o ano de `order_date`
2. Crie a coluna `mes` extraindo o mês de `order_date`
3. Crie a coluna `dias_para_entrega` com `datediff(shipped_date, order_date)`
4. Filtre apenas pedidos que foram enviados (`shipped_date` não nulo)
5. Mostre a média de `dias_para_entrega` por ano

In [0]:
# Exercicio 2 — Solução
exercicio_2 = (
    orders
    .withColumn("ano", F.year(F.col("order_date")))
    .withColumn("mes", F.month(F.col("order_date")))
    .withColumn(
        "dias_para_entrega",
        F.datediff(F.col("shipped_date"), F.col("order_date"))
    )
    .filter(F.col("shipped_date").isNotNull())
)

# Amostra
exercicio_2.select(
    "order_id", "order_date", "shipped_date", "ano", "mes", "dias_para_entrega"
).show(10)

# Média de dias para entrega por ano
exercicio_2.groupBy("ano").agg(
    F.round(F.avg("dias_para_entrega"), 1).alias("media_dias_entrega"),
    F.count("*").alias("total_pedidos")
).orderBy("ano").show()

### Exercício 3 — Padronizar países e tratar nulos

Usando `customers`:
1. Crie a coluna `country_padronizado` com o país em maiúsculo (`upper()`)
2. Crie a coluna `region_tratada` usando `coalesce()` para substituir `NULL` por `"N/A"`
3. Mostre o resultado com as colunas originais e tratadas
4. Conte quantos clientes tinham `region` nula vs preenchida

In [0]:
# Exercicio 3 — Solução
exercicio_3 = (
    customers
    .withColumn("country_padronizado", F.upper(F.col("country")))
    .withColumn("region_tratada", F.coalesce(F.col("region"), F.lit("N/A")))
)

# Amostra do resultado
exercicio_3.select(
    "customer_id", "company_name",
    "country", "country_padronizado",
    "region", "region_tratada"
).show(15, truncate=False)

# Contagem de nulos vs preenchidos
exercicio_3.groupBy(
    F.when(F.col("region").isNull(), "Nulo").otherwise("Preenchido").alias("status_region")
).count().show()

---

## Resumo do Bloco 02

| Operação | PySpark | SQL Equivalente |
|----------|---------|----------------|
| Selecionar colunas | `df.select("col")` | `SELECT col FROM table` |
| Filtrar linhas | `df.filter(cond)` | `WHERE cond` |
| Criar coluna | `df.withColumn("nova", expr)` | `SELECT *, expr AS nova` |
| Renomear | `df.withColumnRenamed("a", "b")` | `SELECT a AS b` |
| Remover coluna | `df.drop("col")` | (não incluir no SELECT) |
| Valores únicos | `df.distinct()` | `SELECT DISTINCT` |
| Condicional | `F.when(cond, val).otherwise(val)` | `CASE WHEN ... THEN ... ELSE ... END` |
| Diferença de datas | `F.datediff(col1, col2)` | `DATEDIFF(col1, col2)` |
| Tratar nulos | `F.coalesce(col, F.lit("default"))` | `COALESCE(col, 'default')` |
| Preencher nulos | `df.fillna({"col": valor})` | (não tem equivalente direto) |

**Próximo bloco:** Agregações com `groupBy().agg()` e Joins entre múltiplas tabelas.